## Transfer learning 
We will use VGG16

- We will learn about transfer learning
- Then we will code 

- Model is trained on one task and is reused for a differnt but related task. Instead of tarining a model from scratch, which can be computationally expensive and require large datasets, transfer learning leverages knowledge from a pre-trained model to improve learning efficiency and performance.

- Pretraining on a Large Dataset
    - A model is first trained on large dataset
    - The model learns general features, such as edges and shapes in images or syntax and semantics in text.
- Fine tuning for new task 
    - The pre trained model is then adapted to a new , smaller dataset
    - Some layers may be frozen (not updated), while others are fine-tuned for the specific task.

 ## Steps 
 -  Import ts 
 - PIL image import - documentation the format should be fine

 ## Transformations
 - Reshape (786)->(28,28) 
 - datatype - np.unit8
 - 1D to 3D - 3,28,28
 - tensor - pil image
 - resize - 3,256,256
 - center crop - 3,224,224
 - tensor (scale) -> (0,1)
 - normalize

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt 
import optuna


In [3]:
torch.manual_seed(42)
device = torch.device('cuda')

In [4]:
df = pd.read_csv(r"C:\Users\pieush\Desktop\Root\Work\Courses\Pytorch\ANN\fashion-mnist_train.csv")
df2 = pd.read_csv(r"C:\Users\pieush\Desktop\Root\Work\Courses\Pytorch\ANN\fashion-mnist_test.csv")

In [5]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values
# test dataset
X_v = df2.iloc[:,1:].values
y_v = df2.iloc[:,0].values

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_valid , y_valid = X_v , y_v

In [8]:
# transformations 
from torchvision.transforms import transforms

custom_transform = transforms.Compose(
    [transforms.Resize(256),
     transforms.CenterCrop(224),
     transforms.ToTensor(),
     transforms.Normalize(mean=[0.485,0.456,0.406], std =[0.229,0.224,0.225])]
)

In [9]:

from PIL import Image
import numpy as np

In [39]:
class CustomDataset(Dataset):
    def __init__(self,features, labels , transform):
        self.features = features
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, index):
        # resize to (28,28)
        image = self.features[index].reshape(28,28)
        #change datatype to np.uint8
        image = image.astype(np.uint8)
        # change bw to color
        image = np.stack([image]*3 , axis = -1)
        #convert array to PIL image 
        image = Image.fromarray(image)
        #apply transform
        image = self.transform(image)
        #return
        return image , torch.tensor(self.labels[index],dtype = torch.long)

In [40]:
train_dataset = CustomDataset(X_train,y_train,transform=custom_transform)
test_dataset = CustomDataset(X_test,y_test,transform=custom_transform)

In [41]:
train_loader = DataLoader(train_dataset, batch_size=32 , shuffle = True , pin_memory= True)
test_loader = DataLoader(test_dataset,batch_size=32 , shuffle= False , pin_memory= True)

In [43]:
# fetch the pretrained model
import torchvision.models as models

vgg16 = models.vgg16(pretrained = True)

c:\Users\pieush\Anaconda\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\pieush\Anaconda\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [42]:
for param in vgg16.features.parameters():
    param.requires_grad = False

In [44]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [45]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512,10)
)

In [46]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [47]:
vgg16 = vgg16.to(device)

In [48]:
learning_rate = 0.001
epochs = 10

In [49]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters() , lr= learning_rate)

In [50]:
for epoch in range(epochs):
        total_epoch_loss = 0
        for batch_features , batch_labels in train_loader:
            # move data to gpu
            batch_labels = batch_labels.type(torch.LongTensor) 
            batch_features,batch_labels = batch_features.to(device) , batch_labels.to(device)
            # forward pass
            outputs = vgg16(batch_features) 
            # calculate loss
            loss = criterion(outputs,batch_labels)   # outputs - Float , batch_labels -> long
            # back pass 
            optimizer.zero_grad()
            loss.backward()
            # update grads
            optimizer.step()
            total_epoch_loss = total_epoch_loss + loss.item()
        avg_loss = total_epoch_loss/len(train_loader)
        print(f'Epoch: {epoch + 1} , Loss: {avg_loss}') 

Epoch: 1 , Loss: 0.44530345249176023


KeyboardInterrupt: 